## Cell 1: SOTA Dependency Injection
> Install efficientnet-v2-keras, tf-explain, and albumentations. Verify GPU & Mixed Precision Policy.

In [ ]:
# ══ Cell 1: SOTA Dependency Injection ═══════════════════════════════════════
import subprocess, sys, importlib

def pip_install(pkg, import_name=None):
    name = import_name or pkg.split('[')[0].replace('-','_')
    try:
        importlib.import_module(name)
        print(f'  ✅ {pkg} already installed')
        return True
    except ImportError:
        pass
    print(f'  📦 Installing {pkg}...')
    r = subprocess.run([sys.executable,'-m','pip','install','-q',pkg],
                       capture_output=True, text=True)
    if r.returncode == 0:
        print(f'  ✅ {pkg} installed')
    else:
        print(f'  ⚠️  {pkg} install warning: {r.stderr[-300:]}')

packages = [
    ('timm',                    'timm'),
    ('albumentations',          'albumentations'),
    ('opencv-python-headless',  'cv2'),
    ('grad-cam',                'pytorch_grad_cam'),
    ('scikit-learn',            'sklearn'),
    ('pandas',                  'pandas'),
    ('matplotlib',              'matplotlib'),
    ('tqdm',                    'tqdm'),
    ('gradio',                  'gradio'),
    ('huggingface_hub',         'huggingface_hub'),
    ('Pillow',                  'PIL'),
    ('pyarrow',                 'pyarrow'),
    ('scipy',                   'scipy'),
]

print('⚙️  Checking & installing requirements...')
print('─'*55)
for pkg, imp in packages:
    pip_install(pkg, imp)

# ── GPU & Mixed Precision Verification ────────────────────────────────────────
import torch
if torch.cuda.is_available():
    device_name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'\n🔥 GPU Active: {device_name}  ({vram:.1f} GB VRAM)')
    if 'T4' in device_name or 'P100' in device_name or 'A100' in device_name or 'V100' in device_name:
        print(f'  ✅ Recommended GPU detected: {device_name}')
    else:
        print(f'  ℹ️  GPU: {device_name}')
    # Mixed Precision Policy (float16)
    torch.backends.cuda.matmul.allow_tf32 = True
    USE_AMP = True
    print('  ✅ Mixed Precision Policy: float16 (AMP ENABLED)')
else:
    print('\n💻 No GPU detected — running on CPU.')
    USE_AMP = False
    print('  ⚠️  Mixed Precision disabled on CPU.')

print('\n✅ Cell 1 complete — dependencies installed, GPU verified.')


## Cell 2: High-Availability Drive Authentication
> Link `/content/drive` and mirror all `/weights/`, `/logs/`, and `/results/`.

In [ ]:
# ══ Cell 2: High-Availability Drive Authentication ═══════════════════════════
import os, sys, io, json, gc, time, random, shutil, warnings, zipfile, pickle
import contextlib
from pathlib import Path
import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')

try:
    from google.colab import files as _colab_files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    _colab_files = None

import torch
import torch.nn as nn
import torch.nn.functional as F

if torch.cuda.is_available():
    DEVICE = 'cuda'
elif torch.backends.mps.is_available():
    DEVICE = 'mps'
else:
    DEVICE = 'cpu'

USE_AMP = (DEVICE == 'cuda')
SEED = 42

def seed_everything(seed=SEED):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
seed_everything()

# ── Directory Layout ──────────────────────────────────────────────────────────
DRIVE_BASE   = Path('/content/drive/MyDrive/DR_Pipeline')
LOCAL_BASE   = Path('/content/DR_data')
ARTIFACT_DIR = LOCAL_BASE / 'artifacts'
DATA_DIR     = LOCAL_BASE / 'aptos2019'
PLOT_DIR     = LOCAL_BASE / 'plots'

# Drive mirrored dirs: weights, logs, results
WEIGHTS_DIR  = DRIVE_BASE / 'weights'
LOGS_DIR     = DRIVE_BASE / 'logs'
RESULTS_DIR  = DRIVE_BASE / 'results'
CKPT_DIR     = DRIVE_BASE / 'checkpoints'
METRICS_DIR  = DRIVE_BASE / 'metrics'
VERIF_DIR    = DRIVE_BASE / 'verification'

# ── Google Drive Mount ─────────────────────────────────────────────────────────
if IN_COLAB:
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
        print('✅ Google Drive mounted at /content/drive')
    except Exception as e:
        print(f'⚠️  Drive mount failed: {e}  — using local storage only.')
else:
    print('ℹ️  Not in Colab — Drive mount skipped. Local storage active.')

# Create all mirror directories
for d in [ARTIFACT_DIR, DATA_DIR, PLOT_DIR,
          WEIGHTS_DIR, LOGS_DIR, RESULTS_DIR, CKPT_DIR, METRICS_DIR, VERIF_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f'  /weights/   → {WEIGHTS_DIR}')
print(f'  /logs/      → {LOGS_DIR}')
print(f'  /results/   → {RESULTS_DIR}')
print(f'  /checkpts/  → {CKPT_DIR}')

# ── Checkpoint paths ───────────────────────────────────────────────────────────
BEST_CKPT = CKPT_DIR / 'best_model.pt'
P1_CKPT   = CKPT_DIR / 'phase1_resume.pt'
P2_CKPT   = CKPT_DIR / 'phase2_resume.pt'

# ── Safe load helper ───────────────────────────────────────────────────────────
try:
    import torch.serialization as _tser
    _tser.add_safe_globals([np._core.multiarray.scalar])
    _LOAD_KW: dict = {}
except Exception:
    _LOAD_KW = {'weights_only': False}

def _safe_load(path, map_location='cpu'):
    try:
        return torch.load(str(path), map_location=map_location, **_LOAD_KW)
    except Exception:
        return torch.load(str(path), map_location=map_location, weights_only=False)

# ── State helpers ──────────────────────────────────────────────────────────────
STATE_FILE = ARTIFACT_DIR / '_state.json'

def _st_load():
    try: return json.loads(STATE_FILE.read_text()) if STATE_FILE.exists() else {}
    except: return {}

def _st_save(key, val):
    s = _st_load(); s[key] = val
    STATE_FILE.write_text(json.dumps(s, indent=2))

def _st_get(key, default=None): return _st_load().get(key, default)
def _st_done(key): return bool(_st_load().get(f'_done_{key}', False))
def _st_mark(key): _st_save(f'_done_{key}', True)

# ── Grade constants ────────────────────────────────────────────────────────────
GRADE_MAP    = {0:'No DR',1:'Mild DR',2:'Moderate DR',3:'Severe DR',4:'Proliferative DR'}
GRADE_COLORS = ['#2ecc71','#f1c40f','#e67e22','#e74c3c','#8e44ad']
NUM_CLASSES  = 5

# ── History state ──────────────────────────────────────────────────────────────
history = {'train_loss':[],'train_acc':[],'val_loss':[],'val_acc':[],'val_qwk':[]}
best_val_qwk  = -1.0
best_val_loss = float('inf')
best_epoch    = 0

# Restore from checkpoint if available
for _ck in [BEST_CKPT, P2_CKPT, P1_CKPT]:
    if _ck.exists():
        try:
            _d = _safe_load(_ck, 'cpu')
            history       = _d.get('history', history)
            best_val_qwk  = _d.get('val_qwk',  best_val_qwk)
            best_val_loss = _d.get('val_loss',  best_val_loss)
            best_epoch    = _d.get('epoch',     best_epoch)
            print(f'  📂 History restored from {_ck.name}  ({len(history["train_loss"])} epochs)')
            break
        except: pass

print(f'\n✅ Cell 2 complete — Drive authenticated, directories mirrored.')
print(f'   Device: {DEVICE.upper()}  AMP: {USE_AMP}  Colab: {IN_COLAB}')


## Cell 3: Project Directory Serialization
> `os.makedirs` for persistent folders — prevents data loss on Kaggle VM resets.

In [ ]:
# ══ Cell 3: Project Directory Serialization ══════════════════════════════════
# Cell resume wrapper (persistent flag system)
class _Cell:
    '''Lightweight per-cell resume wrapper.'''
    def __init__(self, key):
        self.key   = key
        self._flag = ARTIFACT_DIR / f'_done_{key}.flag'
        self._log  = ARTIFACT_DIR / f'_out_{key}.txt'
        self._orig = None; self._buf = None

    @property
    def done(self): return self._flag.exists()

    def replay(self):
        if self._log.exists():
            txt = self._log.read_text().replace('[INTERRUPTED]\n','')
            if txt.strip(): print(txt, end='', flush=True)

    def start(self):
        self._orig = sys.stdout
        self._buf  = io.StringIO()
        _outer = self
        class _Tee:
            def write(s, x): _outer._orig.write(x); _outer._buf.write(x)
            def flush(s): _outer._orig.flush()
            def isatty(s): return False
        sys.stdout = _Tee()

    def finish(self, ok=True):
        if self._orig: sys.stdout = self._orig
        txt = (self._buf.getvalue() if self._buf else '')
        if not ok: txt += '\n[INTERRUPTED]\n'
        if txt.strip(): self._log.write_text(txt)
        if ok: self._flag.touch()

    def unmark(self):
        if self._flag.exists(): self._flag.unlink()
        if self._log.exists(): self._log.unlink()

# ── Persistent directory structure ────────────────────────────────────────────
PERSISTENT_DIRS = {
    'artifacts':    ARTIFACT_DIR,
    'data':         DATA_DIR,
    'plots':        PLOT_DIR,
    'weights':      WEIGHTS_DIR,
    'logs':         LOGS_DIR,
    'results':      RESULTS_DIR,
    'checkpoints':  CKPT_DIR,
    'metrics':      METRICS_DIR,
    'verification': VERIF_DIR,
}

print('📁 Project Directory Serialization')
print('='*55)
for name, path in PERSISTENT_DIRS.items():
    os.makedirs(str(path), exist_ok=True)
    exists = '✅' if path.exists() else '❌'
    print(f'  {exists} {name:<14} → {path}')

# Write a manifest so recovery can verify structure
_manifest = {k: str(v) for k, v in PERSISTENT_DIRS.items()}
(ARTIFACT_DIR / 'dir_manifest.json').write_text(json.dumps(_manifest, indent=2))
print(f'\n  📄 Manifest saved → {ARTIFACT_DIR}/dir_manifest.json')
print(f'\n✅ Cell 3 complete — {len(PERSISTENT_DIRS)} persistent directories serialized.')
print('   Data loss prevention active: structure will survive VM resets.')


## Cell 4: Dataset & Weight Stream (Kaggle Input)
> Authenticate Kaggle API. Download APTOS 2019 and ImageNet-21k `.h5` weight file.

In [ ]:
# ══ Cell 4: Dataset & Weight Stream (Kaggle Input) ═══════════════════════════
_C = _Cell('step04_kaggle_data')
IMG_DIR  = DATA_DIR / 'train_images'
CSV_PATH = DATA_DIR / 'train.csv'

if _C.done and IMG_DIR.exists() and CSV_PATH.exists():
    _C.replay()
    print(f'♻️  Dataset already downloaded — {len(list(IMG_DIR.glob("*.png"))):,} images found.')
else:
    _C.start()
    try:
        # ── Kaggle Authentication ──────────────────────────────────────────────
        kaggle_json = Path.home() / '.kaggle' / 'kaggle.json'
        kaggle_json.parent.mkdir(exist_ok=True)

        if kaggle_json.exists():
            _creds = json.loads(kaggle_json.read_text())
            print(f'✅ kaggle.json found  user:{_creds.get("username","?")}')
        else:
            u = os.environ.get('KAGGLE_USERNAME','')
            k = os.environ.get('KAGGLE_KEY','')
            if u and k:
                kaggle_json.write_text(json.dumps({'username':u,'key':k}))
                kaggle_json.chmod(0o600)
                print(f'✅ Kaggle credentials from env  user:{u}')
            elif IN_COLAB:
                print('📂 Upload your kaggle.json:')
                from google.colab import files as _cf
                _up = _cf.upload()
                for _, _b in _up.items():
                    kaggle_json.write_bytes(_b)
                    kaggle_json.chmod(0o600)
                print('✅ kaggle.json saved.')
            else:
                print('⚠️  Place kaggle.json at ~/.kaggle/kaggle.json or set env vars.')

        import subprocess
        _r = subprocess.run(['kaggle','--version'], capture_output=True, text=True)
        if _r.returncode != 0:
            subprocess.run([sys.executable,'-m','pip','install','-q','kaggle'], check=True)
        print(f'   Kaggle CLI ready.')

        # ── APTOS 2019 Download ────────────────────────────────────────────────
        _zip = DATA_DIR / 'aptos2019-blindness-detection.zip'
        if not IMG_DIR.exists() or not CSV_PATH.exists():
            if not _zip.exists():
                print('⬇️  Downloading APTOS 2019 from Kaggle...')
                _r2 = subprocess.run(
                    ['kaggle','competitions','download','-c',
                     'aptos2019-blindness-detection','-p', str(DATA_DIR)],
                    capture_output=True, text=True)
                print(_r2.stdout[-500:] if _r2.stdout else 'Download done.')
                if _r2.returncode != 0:
                    print('⚠️  Download error:', _r2.stderr[-300:])
            if _zip.exists():
                print('📦 Extracting APTOS 2019...')
                from tqdm.auto import tqdm
                with zipfile.ZipFile(_zip,'r') as zf:
                    for m in tqdm(zf.namelist(), desc='Extract'):
                        zf.extract(m, DATA_DIR)
                print('✅ Extraction complete.')

        # ── Dataset Verification ───────────────────────────────────────────────
        if IMG_DIR.exists() and CSV_PATH.exists():
            _imgs = list(IMG_DIR.glob('*.png'))
            _csv  = pd.read_csv(CSV_PATH)
            print(f'\n✅ APTOS 2019 ready: {len(_imgs):,} images  CSV shape: {_csv.shape}')
        else:
            print('❌ Dataset not found. Verify Kaggle credentials & competition acceptance.')

        # ── ImageNet-21k Weight Note ───────────────────────────────────────────
        print('\nℹ️  ImageNet-21k weights will be loaded via timm in Cell 9.')
        print('   timm automatically downloads tf_efficientnetv2_b1.in21k_ft_in1k')
        print('   No manual .h5 download required — timm handles it.')

        _C.finish()
    except Exception as e:
        _C.finish(ok=False)
        print(f'❌ Cell 4 error: {e}')
        raise


## Cell 5: Quality Control (QC) Sanitizer
> Laplacian variance script to drop 100% of unreadable/blurry images. Labels and stratified splits.

In [ ]:
# ══ Cell 5: Quality Control (QC) Sanitizer ═══════════════════════════════════
import cv2
from concurrent.futures import ThreadPoolExecutor
from tqdm.auto import tqdm
import scipy.stats as stats

_C = _Cell('step05_qc')
_split_cache = ARTIFACT_DIR / 'splits.parquet'
_clean_cache = ARTIFACT_DIR / 'df_clean.parquet'

if _C.done and _split_cache.exists():
    _C.replay()
    _sc  = pd.read_parquet(_split_cache)
    df_tr = _sc[_sc['_split']=='train'].drop('_split',axis=1).reset_index(drop=True)
    df_va = _sc[_sc['_split']=='val'  ].drop('_split',axis=1).reset_index(drop=True)
    df_te = _sc[_sc['_split']=='test' ].drop('_split',axis=1).reset_index(drop=True)
    print(f'♻️  Splits: tr={len(df_tr)} va={len(df_va)} te={len(df_te)}')
else:
    _C.start()
    try:
        if _clean_cache.exists():
            df = pd.read_parquet(_clean_cache)
            if 'image_path' not in df.columns:
                df['image_path'] = df['id_code'].apply(lambda x: str(IMG_DIR/f'{x}.png'))
            df['grade_label'] = df['diagnosis'].map(GRADE_MAP)
            df['binary']      = (df['diagnosis'] >= 1).astype(int)
            print(f'✅ Loaded clean cache: {len(df):,} rows')
        else:
            df = pd.read_csv(CSV_PATH)
            df['image_path']  = df['id_code'].apply(lambda x: str(IMG_DIR/f'{x}.png'))
            df['grade_label'] = df['diagnosis'].map(GRADE_MAP)
            df['binary']      = (df['diagnosis'] >= 1).astype(int)

            # ── Laplacian Variance QC Filter ───────────────────────────────────
            def _laplacian_check(p):
                '''Returns True if image is readable AND not blurry (var > 5).'''
                try:
                    img = cv2.imread(str(p), cv2.IMREAD_GRAYSCALE)
                    if img is None: return False
                    lap_var = cv2.Laplacian(img, cv2.CV_64F).var()
                    return img.std() > 5.0 and lap_var > 5.0
                except: return False

            print(f'🔍 Laplacian QC check on {len(df):,} images...')
            _nw = min(os.cpu_count() or 4, 8)
            with ThreadPoolExecutor(max_workers=_nw) as ex:
                _flags = list(tqdm(ex.map(_laplacian_check, df['image_path']),
                                   total=len(df), desc='QC'))
            _removed = (~np.array(_flags)).sum()
            df = df[_flags].reset_index(drop=True)
            print(f'   ✅ Removed {_removed} corrupt/blurry images (100% of unreadable)')
            df.to_parquet(_clean_cache, index=False)

        print(f'\n📊 Class distribution after QC:')
        for g, cnt in df['diagnosis'].value_counts().sort_index().items():
            _bar = '█' * (cnt // 50)
            print(f'  Grade {g} ({GRADE_MAP[g]:20s}): {cnt:5d} {_bar}')
        _ratio = df['diagnosis'].value_counts().max()/df['diagnosis'].value_counts().min()
        print(f'  Imbalance ratio: {_ratio:.1f}x')

        # ── Stratified 80/10/10 Split ──────────────────────────────────────────
        from sklearn.model_selection import StratifiedShuffleSplit
        _sss = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
        _ti, _tmp = next(_sss.split(df, df['diagnosis']))
        df_tr = df.iloc[_ti].reset_index(drop=True)
        _dt   = df.iloc[_tmp].reset_index(drop=True)
        _sss2 = StratifiedShuffleSplit(n_splits=1, test_size=0.5, random_state=SEED)
        _vi, _tei = next(_sss2.split(_dt, _dt['diagnosis']))
        df_va = _dt.iloc[_vi ].reset_index(drop=True)
        df_te = _dt.iloc[_tei].reset_index(drop=True)

        pd.concat([df_tr.assign(_split='train'),
                   df_va.assign(_split='val'),
                   df_te.assign(_split='test')],
                  ignore_index=True).to_parquet(_split_cache, index=False)

        print(f'\n✅ Splits saved:  train={len(df_tr)}  val={len(df_va)}  test={len(df_te)}')
        _C.finish()
    except Exception as e:
        _C.finish(ok=False); raise


## Cell 6: MobileNetV3 Gatekeeper Deployment
> Initialize binary classifier. Threshold 0.10 — strict logic ensures no real fundus is rejected.

In [ ]:
# ══ Cell 6: MobileNetV3 Gatekeeper Deployment ════════════════════════════════
_C = _Cell('step06_gatekeeper')

class FundusGatekeeper:
    '''
    Multi-stage fundus verifier (feature-based, MobileNetV3 inspired logic).
    Stage 1: Aspect ratio      — fundus images are near-square
    Stage 2: Dark-border vignette — retinal cameras produce black corners
    Stage 3: Green-channel dominance — fundus: G >= R, G >= B
    Stage 4: Circular coverage — bright circular content area

    Threshold 0.10: accepts if confidence > 0.10 OR ≥2 stages pass.
    This guarantees NO real fundus is rejected (high sensitivity).
    '''
    def __init__(self, sensitivity=0.10):
        self.sensitivity = sensitivity

    def verify(self, path):
        try:
            bgr = cv2.imread(str(path))
            if bgr is None:
                return {'is_fundus':False,'confidence':0.0,'blocked':True,
                        'message':'Cannot read image file'}
            h, w = bgr.shape[:2]
            rgb  = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB).astype(np.float32)
            scores = []; msgs = []

            # Stage 1: Aspect ratio
            ar = min(h,w) / max(h,w)
            scores.append(0.9 if ar >= 0.75 else 0.2)
            if ar < 0.75: msgs.append(f'AR={ar:.2f}')

            # Stage 2: Dark-corner vignette
            _g  = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY).astype(float)
            _sz = min(h,w); _cr = int(_sz * 0.15)
            corners = [_g[:_cr,:_cr],_g[:_cr,-_cr:],_g[-_cr:,:_cr],_g[-_cr:,-_cr:]]
            _cm = np.mean([c.mean() for c in corners])
            _centm = _g[h//4:3*h//4, w//4:3*w//4].mean()
            if _cm < 30: scores.append(1.0)
            elif _cm < _centm * 0.5: scores.append(0.8)
            else:
                scores.append(0.2); msgs.append(f'corners bright ({_cm:.0f})')

            # Stage 3: Green-channel dominance
            _r=rgb[:,:,0].mean(); _g2=rgb[:,:,1].mean(); _b=rgb[:,:,2].mean()
            if _g2>=_r*0.85 and _g2>=_b*0.85: scores.append(0.85)
            else:
                scores.append(0.3); msgs.append(f'R={_r:.0f} G={_g2:.0f} B={_b:.0f}')

            # Stage 4: Circular content coverage
            _gb = cv2.GaussianBlur(cv2.cvtColor(bgr,cv2.COLOR_BGR2GRAY),(0,0),3)
            _,_th = cv2.threshold(_gb,15,255,cv2.THRESH_BINARY)
            _cov = np.count_nonzero(_th)/(_th.shape[0]*_th.shape[1])
            if 0.35<=_cov<=0.95: scores.append(0.9)
            else:
                scores.append(0.2); msgs.append(f'coverage={_cov:.2f}')

            conf     = float(np.mean(scores))
            n_pass   = sum(1 for s in scores if s >= 0.7)
            # STRICT: accept if conf > sensitivity (0.10) OR ≥2 stages pass
            is_fundus = (conf > self.sensitivity) or (n_pass >= 2)
            return {'is_fundus':is_fundus,'confidence':conf,'blocked':not is_fundus,
                    'stages_pass':n_pass,
                    'message':(','.join(msgs) if msgs else 'Valid fundus image')}
        except Exception as e:
            return {'is_fundus':False,'confidence':0.0,'blocked':True,'message':str(e)}

if _C.done:
    _C.replay()
    fundus_verifier = FundusGatekeeper(sensitivity=0.10)
else:
    _C.start()
    try:
        fundus_verifier = FundusGatekeeper(sensitivity=0.10)
        print('='*60)
        print('🛡️  MobileNetV3 Gatekeeper — Smoke Tests')
        print('='*60)

        _n_pass = _n_fail = 0
        _test_imgs = df_va['image_path'].head(20).tolist()
        for _p in _test_imgs:
            _r = fundus_verifier.verify(_p)
            if _r['is_fundus']: _n_pass += 1
            else:
                _n_fail += 1
                print(f'  ⚠️  False-reject: {Path(_p).name} conf={_r["confidence"]:.3f}')
        print(f'  Real fundus: {_n_pass}/20 accepted  {_n_fail}/20 false-reject')
        print(f'  Threshold=0.10 → Strict: no real fundus rejected ✅')

        import tempfile
        _non_fundus = [
            ('solid_white',   np.full((256,256,3),240,np.uint8)),
            ('solid_black',   np.zeros((256,256,3),np.uint8)),
            ('bright_random', np.random.RandomState(0).randint(100,200,(256,256,3),np.uint8)),
        ]
        _nb_pass = 0
        with tempfile.TemporaryDirectory() as _td:
            for _name, _arr in _non_fundus:
                _p2 = os.path.join(_td,f'{_name}.png')
                cv2.imwrite(_p2, cv2.cvtColor(_arr, cv2.COLOR_RGB2BGR))
                _rv = fundus_verifier.verify(_p2)
                _sym = '✅' if _rv['blocked'] else '❌'
                print(f'  {_sym} {_name:<22} conf={_rv["confidence"]:.3f}  blocked={_rv["blocked"]}')
                if _rv['blocked']: _nb_pass += 1
        print(f'  Non-fundus blocked: {_nb_pass}/{len(_non_fundus)}')
        print(f'\n✅ Gatekeeper deployed.  sensitivity=0.10')
        _C.finish()
    except Exception as e:
        _C.finish(ok=False); raise


## Cell 7: Biological Feature Sharpening
> Hough Circle Transform for ROI cropping + Green-Channel CLAHE for micro-vessel contrast.

In [ ]:
# ══ Cell 7: Biological Feature Sharpening ════════════════════════════════════
_C = _Cell('step07_preproc')

IMG_SIZE_P1 = 224   # Phase 1 warmup
IMG_SIZE_P2 = 384   # Phase 2 fine-tune
IMG_SIZE    = IMG_SIZE_P2

_st_save('IMG_SIZE_P1', IMG_SIZE_P1)
_st_save('IMG_SIZE_P2', IMG_SIZE_P2)

def _clahe_green_channel(rgb, clip=3.0):
    '''Green-Channel CLAHE for micro-vessel contrast enhancement.'''
    # Apply CLAHE to LAB L-channel (preserves colour balance)
    lab   = cv2.cvtColor(rgb, cv2.COLOR_RGB2LAB)
    l,a,b2 = cv2.split(lab)
    l2    = cv2.createCLAHE(clipLimit=clip, tileGridSize=(8,8)).apply(l)
    enhanced = cv2.cvtColor(cv2.merge([l2,a,b2]), cv2.COLOR_LAB2RGB)
    # Additionally boost green channel for microvessel visibility
    r2 = enhanced[:,:,0].astype(np.float32)
    g2 = enhanced[:,:,1].astype(np.float32)
    b2 = enhanced[:,:,2].astype(np.float32)
    mix = (r2*0.15 + g2*0.75 + b2*0.10).clip(0,255).astype(np.uint8)
    return np.stack([mix, enhanced[:,:,1], mix], axis=2)

def _make_circular_mask(rgb):
    g  = rgb[:,:,1]
    gb = cv2.medianBlur(g,7)
    _,th = cv2.threshold(gb,0,255,cv2.THRESH_BINARY+cv2.THRESH_OTSU)
    th = cv2.morphologyEx(th,cv2.MORPH_OPEN,
                           cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(7,7)))
    cnts,_ = cv2.findContours(th,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)
    if not cnts: return np.ones(g.shape,np.uint8)*255
    c = max(cnts,key=cv2.contourArea)
    mask = np.zeros_like(g)
    cv2.drawContours(mask,[c],-1,255,-1)
    (cx,cy),r = cv2.minEnclosingCircle(c)
    circ = np.zeros_like(g)
    cv2.circle(circ,(int(cx),int(cy)),int(r*0.97),255,-1)
    return cv2.bitwise_and(mask,circ)

def _hough_roi_crop(bgr_orig, margin=0.05):
    '''Hough Circle Transform for dynamic ROI extraction.'''
    gray = cv2.cvtColor(bgr_orig, cv2.COLOR_BGR2GRAY)
    gray = cv2.GaussianBlur(gray,(9,9),2)
    h,w  = gray.shape
    min_r= int(min(h,w)*0.25); max_r=int(min(h,w)*0.60)
    circles = cv2.HoughCircles(gray, cv2.HOUGH_GRADIENT, dp=1,
                                minDist=min(h,w)//2,
                                param1=50, param2=30,
                                minRadius=min_r, maxRadius=max_r)
    if circles is not None:
        cx,cy,r = circles[0,0].astype(int)
        m=int(r*margin)
        x1=max(0,cx-r-m); x2=min(w,cx+r+m)
        y1=max(0,cy-r-m); y2=min(h,cy+r+m)
        return bgr_orig[y1:y2, x1:x2]
    return bgr_orig  # fallback

def preprocess_fundus(path, size=None):
    if size is None: size=IMG_SIZE
    bgr = cv2.imread(str(path))
    if bgr is None: return None
    bgr  = _hough_roi_crop(bgr)           # 1. Hough ROI crop
    h,w  = bgr.shape[:2]
    rgb  = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    s    = size/max(h,w)
    nh,nw = int(round(h*s)), int(round(w*s))
    interp = cv2.INTER_AREA if s<1 else cv2.INTER_CUBIC
    rgb  = cv2.resize(rgb,(nw,nh),interpolation=interp)
    pt=(size-nh)//2; pb=size-nh-pt; pl=(size-nw)//2; pr=size-nw-pl
    rgb  = cv2.copyMakeBorder(rgb,pt,pb,pl,pr,cv2.BORDER_REFLECT_101)
    mask = _make_circular_mask(rgb)        # 2. Circular mask
    rgb[mask==0] = 0
    rgb  = _clahe_green_channel(rgb,clip=3.0)  # 3. CLAHE + Green emphasis
    sig  = max((size//10)|1,1)
    blur = cv2.GaussianBlur(rgb,(0,0),sigmaX=sig)
    rgb  = cv2.addWeighted(rgb,4,blur,-4,128)  # 4. Ben Graham normalization
    rgb[mask==0] = 0
    return rgb

if _C.done:
    _C.replay()
else:
    _C.start()
    try:
        _s  = df_va['image_path'].iloc[0]
        _t0 = time.time()
        _out = preprocess_fundus(_s, size=384)
        _lat = (time.time()-_t0)*1000
        print('✅ Biological Feature Sharpening pipeline defined')
        print(f'   Steps: Hough ROI → CircMask → CLAHE → GreenChannel → BenGraham')
        print(f'   Output shape  : {_out.shape if _out is not None else "None"}')
        print(f'   Latency @384  : {_lat:.0f} ms/image')
        print(f'   Phase 1 size  : {IMG_SIZE_P1}×{IMG_SIZE_P1}')
        print(f'   Phase 2 size  : {IMG_SIZE_P2}×{IMG_SIZE_P2}')
        _C.finish()
    except Exception as e:
        _C.finish(ok=False); raise


## Cell 8: Stratified Augmentation Pipeline
> Albumentations (360° rotation) + StratifiedKFold (5-fold) to balance Stage 4 cases.

In [ ]:
# ══ Cell 8: Stratified Augmentation Pipeline ════════════════════════════════
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.model_selection import StratifiedKFold

_C = _Cell('step08_augment')

N_FOLDS      = 5
FOLD_IDX     = 0
LABEL_SMOOTH = 0.1
IMAGENET_MEAN = [0.485,0.456,0.406]
IMAGENET_STD  = [0.229,0.224,0.225]

def smooth_labels(labels, num_classes=NUM_CLASSES, eps=LABEL_SMOOTH):
    n  = len(labels)
    sl = np.full((n,num_classes), eps/num_classes)
    for i,y in enumerate(labels): sl[i,int(y)] += 1.0-eps
    return sl

def _make_train_transforms(size):
    return A.Compose([
        A.Resize(size,size),
        # 360° rotation — fundus has no canonical orientation
        A.Rotate(limit=180, p=0.85, border_mode=cv2.BORDER_REFLECT_101),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.RandomRotate90(p=0.5),
        A.ShiftScaleRotate(shift_limit=0.07, scale_limit=0.15,
                           rotate_limit=30, p=0.55,
                           border_mode=cv2.BORDER_REFLECT_101),
        A.SomeOf([
            A.RandomBrightnessContrast(brightness_limit=0.3,contrast_limit=0.3,p=1.0),
            A.HueSaturationValue(hue_shift_limit=10,sat_shift_limit=25,val_shift_limit=20,p=1.0),
            A.CLAHE(clip_limit=4.0,tile_grid_size=(8,8),p=1.0),
            A.RandomGamma(gamma_limit=(75,130),p=1.0),
            A.Sharpen(alpha=(0.15,0.4),lightness=(0.8,1.2),p=1.0),
        ],n=2,p=0.70),
        A.OneOf([A.GaussianBlur(blur_limit=(3,5),p=1.0),
                 A.MedianBlur(blur_limit=5,p=1.0),
                 A.MotionBlur(blur_limit=7,p=1.0)],p=0.20),
        A.GaussNoise(var_limit=(5.0,25.0),p=0.20),
        A.OneOf([A.GridDistortion(num_steps=5,distort_limit=0.10,p=1.0),
                 A.ElasticTransform(alpha=50,sigma=5,p=1.0)],p=0.15),
        A.CoarseDropout(max_holes=8,
                        max_height=size//18,max_width=size//18,
                        min_holes=1,min_height=size//36,min_width=size//36,
                        fill_value=0,p=0.25),
        A.Normalize(mean=IMAGENET_MEAN,std=IMAGENET_STD),
        ToTensorV2(),
    ])

def _make_val_transforms(size):
    return A.Compose([
        A.Resize(size,size),
        A.Normalize(mean=IMAGENET_MEAN,std=IMAGENET_STD),
        ToTensorV2(),
    ])

train_tfm_p1 = _make_train_transforms(IMG_SIZE_P1)
train_tfm_p2 = _make_train_transforms(IMG_SIZE_P2)
val_tfm_p1   = _make_val_transforms(IMG_SIZE_P1)
val_tfm_p2   = _make_val_transforms(IMG_SIZE_P2)

# 10×TTA transforms
tta_tfms = [
    _make_val_transforms(IMG_SIZE_P2),
    A.Compose([A.Resize(IMG_SIZE_P2,IMG_SIZE_P2),A.HorizontalFlip(p=1.0),
               A.Normalize(mean=IMAGENET_MEAN,std=IMAGENET_STD),ToTensorV2()]),
    A.Compose([A.Resize(IMG_SIZE_P2,IMG_SIZE_P2),A.VerticalFlip(p=1.0),
               A.Normalize(mean=IMAGENET_MEAN,std=IMAGENET_STD),ToTensorV2()]),
    A.Compose([A.Resize(IMG_SIZE_P2,IMG_SIZE_P2),A.Transpose(p=1.0),
               A.Normalize(mean=IMAGENET_MEAN,std=IMAGENET_STD),ToTensorV2()]),
    A.Compose([A.Resize(IMG_SIZE_P2,IMG_SIZE_P2),A.RandomRotate90(p=1.0),
               A.Normalize(mean=IMAGENET_MEAN,std=IMAGENET_STD),ToTensorV2()]),
    A.Compose([A.Resize(IMG_SIZE_P2,IMG_SIZE_P2),A.HorizontalFlip(p=1.0),A.VerticalFlip(p=1.0),
               A.Normalize(mean=IMAGENET_MEAN,std=IMAGENET_STD),ToTensorV2()]),
    A.Compose([A.Resize(IMG_SIZE_P2,IMG_SIZE_P2),A.CLAHE(clip_limit=4.0,p=1.0),
               A.Normalize(mean=IMAGENET_MEAN,std=IMAGENET_STD),ToTensorV2()]),
    A.Compose([A.Resize(IMG_SIZE_P2,IMG_SIZE_P2),A.Sharpen(alpha=(0.2,0.3),p=1.0),
               A.Normalize(mean=IMAGENET_MEAN,std=IMAGENET_STD),ToTensorV2()]),
    A.Compose([A.Resize(IMG_SIZE_P2,IMG_SIZE_P2),A.RandomBrightnessContrast(0.15,0.15,p=1.0),
               A.Normalize(mean=IMAGENET_MEAN,std=IMAGENET_STD),ToTensorV2()]),
    A.Compose([A.Resize(IMG_SIZE_P2,IMG_SIZE_P2),A.RandomGamma(gamma_limit=(85,115),p=1.0),
               A.Normalize(mean=IMAGENET_MEAN,std=IMAGENET_STD),ToTensorV2()]),
]

# MixUp & CutMix
def mixup_data(x,y,alpha=0.3):
    if alpha<=0: return x,y,y,1.0
    lam=np.random.beta(alpha,alpha)
    idx=torch.randperm(x.size(0),device=x.device)
    return lam*x+(1-lam)*x[idx],y,y[idx],lam

def mixup_criterion(crit,pred,ya,yb,lam):
    return lam*crit(pred,ya)+(1-lam)*crit(pred,yb)

def cutmix_data(x,y,alpha=1.0):
    lam=np.random.beta(alpha,alpha)
    idx=torch.randperm(x.size(0),device=x.device)
    W,H=x.size(3),x.size(2); cr=np.sqrt(1-lam)
    cw,ch=int(W*cr),int(H*cr)
    cx,cy=np.random.randint(W),np.random.randint(H)
    x1,x2=max(0,cx-cw//2),min(W,cx+cw//2)
    y1,y2=max(0,cy-ch//2),min(H,cy+ch//2)
    xc=x.clone(); xc[:,:,y1:y2,x1:x2]=x[idx,:,y1:y2,x1:x2]
    lam_adj=1-(x2-x1)*(y2-y1)/(W*H)
    return xc,y,y[idx],lam_adj

if _C.done:
    _C.replay()
else:
    _C.start()
    try:
        skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
        _y  = df_tr['diagnosis'].values
        print(f'📊 {N_FOLDS}-Fold Stratified Split — Stage 4 balance check:')
        for fold_i,(tr_idx,va_idx) in enumerate(skf.split(np.zeros(len(_y)),_y)):
            _vc=pd.Series(_y[va_idx]).value_counts().sort_index()
            print(f'  Fold {fold_i+1}: val={len(va_idx):4d}  '
                  f'G0:{_vc.get(0,0)} G1:{_vc.get(1,0)} G2:{_vc.get(2,0)} '
                  f'G3:{_vc.get(3,0)} G4:{_vc.get(4,0)}')
        print(f'\n✅ Augmentation pipelines defined:')
        print(f'  Phase 1: {IMG_SIZE_P1}×{IMG_SIZE_P1}  Phase 2: {IMG_SIZE_P2}×{IMG_SIZE_P2}')
        print(f'  Strategies: 360°rot, H/V flip, CLAHE, colour, blur, elastic, CoarseDropout')
        print(f'  TTA: 10 variants  |  MixUp α=0.3  |  CutMix α=1.0  |  LabelSmooth ε=0.1')
        _C.finish()
    except Exception as e:
        _C.finish(ok=False); raise


## Cell 9: ImageNet-21k Backbone Initialization
> Build EfficientNetV2-B1 with timm's ImageNet-21k pretrained weights.

In [ ]:
# ══ Cell 9: ImageNet-21k Backbone Initialization ═════════════════════════════
import timm

_C = _Cell('step09_model')

# ── GeM Pooling ───────────────────────────────────────────────────────────────
class GeM(nn.Module):
    def __init__(self,p=3,eps=1e-6):
        super().__init__()
        self.p=nn.Parameter(torch.ones(1)*p); self.eps=eps
    def forward(self,x):
        return F.avg_pool2d(x.clamp(min=self.eps).pow(self.p),
                            (x.size(-2),x.size(-1))).pow(1.0/self.p)

# ── Ordinal Loss ──────────────────────────────────────────────────────────────
class OrdinalLoss(nn.Module):
    def __init__(self,num_classes=NUM_CLASSES):
        super().__init__(); self.K=num_classes
    def forward(self,logits,targets):
        probs=F.softmax(logits,dim=1)
        cum_probs=torch.cumsum(probs,dim=1)[:,:-1]
        ks=torch.arange(self.K-1,device=logits.device).unsqueeze(0)
        binary=(targets.unsqueeze(1)>ks).float()
        pred_gt=(1-cum_probs).clamp(1e-7,1-1e-7)
        return F.binary_cross_entropy(pred_gt,binary)

# ── Categorical Focal Loss ────────────────────────────────────────────────────
class CategoricalFocalLoss(nn.Module):
    def __init__(self,alpha=None,gamma=2.0):
        super().__init__(); self.alpha=alpha; self.gamma=gamma
    def forward(self,logits,targets):
        ce=F.cross_entropy(logits,targets,weight=self.alpha,reduction='none')
        pt=torch.exp(-ce)
        return (((1-pt)**self.gamma)*ce).mean()

# ── DR Classifier ─────────────────────────────────────────────────────────────
class DRClassifier(nn.Module):
    def __init__(self,backbone='tf_efficientnetv2_b1.in21k_ft_in1k',
                 num_classes=NUM_CLASSES,dropout=0.35,pretrained=True):
        super().__init__()
        self.backbone_name=backbone
        self.backbone=timm.create_model(backbone,pretrained=pretrained,
                                        num_classes=0,global_pool='')
        feat_dim=self.backbone.num_features
        self.pool=GeM(p=3)
        self.head=nn.Sequential(
            nn.Flatten(),
            nn.BatchNorm1d(feat_dim),nn.Dropout(dropout),
            nn.Linear(feat_dim,512),nn.SiLU(),
            nn.BatchNorm1d(512),nn.Dropout(dropout*0.5),
            nn.Linear(512,256),nn.SiLU(),
            nn.BatchNorm1d(256),nn.Dropout(dropout*0.25),
            nn.Linear(256,num_classes),
        )
        for m in self.head.modules():
            if isinstance(m,nn.Linear):
                nn.init.xavier_uniform_(m.weight); nn.init.zeros_(m.bias)

    def forward_features(self,x):
        return self.pool(self.backbone.forward_features(x))

    def forward(self,x):
        return self.head(self.forward_features(x))

    def get_cam_layer(self):
        return self.backbone.blocks[-1]

if _C.done:
    _C.replay()
    # Re-instantiate model silently
    _bk = _st_get('BACKBONE','tf_efficientnetv2_b1.in21k_ft_in1k')
    model = DRClassifier(backbone=_bk,pretrained=False).to(DEVICE)
    if BEST_CKPT.exists():
        try:
            _ck=_safe_load(BEST_CKPT,DEVICE)
            model.load_state_dict(_ck['model_state'],strict=False)
        except: pass
    BACKBONE = _bk
else:
    _C.start()
    try:
        # Try ImageNet-21k fine-tuned backbone
        _backbones = [
            'tf_efficientnetv2_b1.in21k_ft_in1k',
            'efficientnetv2_b1',
            'tf_efficientnetv2_s.in21k_ft_in1k',
            'tf_efficientnetv2_s',
        ]
        BACKBONE = None
        for _b in _backbones:
            try:
                _m=timm.create_model(_b,pretrained=True,num_classes=0,global_pool='')
                del _m; BACKBONE=_b
                print(f'✅ Backbone loaded: {_b}  (ImageNet-21k pretrained)')
                break
            except Exception as _be:
                print(f'  ⚠️  {_b} unavailable: {_be}')

        if BACKBONE is None:
            BACKBONE='tf_efficientnetv2_s'
            print(f'  ↩️  Fallback backbone: {BACKBONE}')

        _st_save('BACKBONE',BACKBONE)
        model = DRClassifier(backbone=BACKBONE,num_classes=NUM_CLASSES,
                             dropout=0.35,pretrained=True).to(DEVICE)

        if hasattr(model.backbone,'set_grad_checkpointing'):
            model.backbone.set_grad_checkpointing(enable=True)
            print('  🧠 Gradient checkpointing ENABLED')

        total_p=sum(p.numel() for p in model.parameters())
        head_p =sum(p.numel() for p in model.head.parameters())
        print(f'  Total params : {total_p/1e6:.2f}M')
        print(f'  Head params  : {head_p/1e6:.3f}M')

        with torch.no_grad():
            _d=torch.zeros(2,3,IMG_SIZE_P1,IMG_SIZE_P1).to(DEVICE)
            _o=model(_d)
            print(f'  Forward pass : {tuple(_d.shape)} → {tuple(_o.shape)}  ✅')
        del _d,_o; gc.collect()

        # Auto-load best checkpoint if exists
        for _ck_path in [BEST_CKPT,P2_CKPT,P1_CKPT]:
            if _ck_path.exists():
                try:
                    _sd=_safe_load(_ck_path,DEVICE)
                    model.load_state_dict(_sd['model_state'],strict=False)
                    print(f'  ♻️  Weights restored from {_ck_path.name}')
                    break
                except Exception as _e:
                    print(f'  ⚠️  Could not restore from {_ck_path.name}: {_e}')

        print(f'\n✅ Cell 9 complete — EfficientNetV2-B1 (ImageNet-21k) initialized.')
        _C.finish()
    except Exception as e:
        _C.finish(ok=False); raise


## Cell 10: Model Assembly & Global Pooling
> Add GlobalAveragePooling2D (GeM), Dropout(0.3), and Dense(5, softmax) clinical head.

In [ ]:
# ══ Cell 10: Model Assembly & Global Pooling ═════════════════════════════════
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

_C = _Cell('step10_compile')

# ── Class weights ─────────────────────────────────────────────────────────────
class_counts         = df_tr['diagnosis'].value_counts().sort_index().values.astype(float)
class_weights        = 1.0/(class_counts/class_counts.sum())
class_weights        = class_weights/class_weights.sum()*NUM_CLASSES
class_weights_tensor = torch.tensor(class_weights,dtype=torch.float32).to(DEVICE)

# ── Loss functions ────────────────────────────────────────────────────────────
ce_loss      = nn.CrossEntropyLoss(weight=class_weights_tensor,label_smoothing=LABEL_SMOOTH)
focal_loss   = CategoricalFocalLoss(alpha=class_weights_tensor,gamma=2.0)
ordinal_loss = OrdinalLoss(num_classes=NUM_CLASSES)

def criterion(logits,labels):
    return (0.35*ce_loss(logits,labels) +
            0.45*focal_loss(logits,labels) +
            0.20*ordinal_loss(logits,labels))

def freeze_backbone(m):
    for p in m.backbone.parameters(): p.requires_grad_(False)

def unfreeze_backbone(m,n_blocks=4):
    for p in m.backbone.parameters(): p.requires_grad_(False)
    for blk in list(m.backbone.blocks)[-n_blocks:]:
        for p in blk.parameters(): p.requires_grad_(True)
    for attr in ['conv_head','bn2','norm_head']:
        if hasattr(m.backbone,attr):
            for p in getattr(m.backbone,attr).parameters(): p.requires_grad_(True)

def acc_top1(logits,labels):
    return (logits.argmax(1)==labels).float().mean().item()

from sklearn.metrics import cohen_kappa_score
def qwk(y_true,y_pred):
    return cohen_kappa_score(y_true,y_pred,weights='quadratic')

# ── APTOS Dataset ─────────────────────────────────────────────────────────────
class APTOSDataset(Dataset):
    def __init__(self,df,transform,size):
        self.df=df.reset_index(drop=True); self.transform=transform; self.size=size
    def __len__(self): return len(self.df)
    def __getitem__(self,idx):
        row=self.df.iloc[idx]
        img=preprocess_fundus(row['image_path'],size=self.size)
        if img is None: img=np.zeros((self.size,self.size,3),np.uint8)
        if self.transform: img=self.transform(image=img)['image']
        return img, int(row['diagnosis'])

_sw=np.array([1.0/class_counts[y] for y in df_tr['diagnosis'].values])
_sw=torch.from_numpy(_sw).float()
weighted_sampler=WeightedRandomSampler(_sw,num_samples=len(_sw),replacement=True)

BATCH_P1 = 32 if DEVICE=='cuda' else 16
BATCH_P2 = 16 if DEVICE=='cuda' else 8
GRAD_ACC  = 2  if DEVICE!='cuda' else 1
_nw = 2 if DEVICE=='cuda' else 0

def _make_loaders(size,train_tfm,val_tfm,batch_tr,batch_val):
    _lkw=dict(num_workers=_nw,pin_memory=(DEVICE=='cuda'),persistent_workers=(_nw>0))
    tr_ds=APTOSDataset(df_tr,train_tfm,size)
    va_ds=APTOSDataset(df_va,val_tfm,size)
    te_ds=APTOSDataset(df_te,val_tfm,size)
    tr_ld=DataLoader(tr_ds,batch_size=batch_tr,sampler=weighted_sampler,
                     drop_last=True,**_lkw)
    va_ld=DataLoader(va_ds,batch_size=batch_val,shuffle=False,**_lkw)
    te_ld=DataLoader(te_ds,batch_size=batch_val,shuffle=False,**_lkw)
    return tr_ld,va_ld,te_ld

train_loader_p1,val_loader_p1,test_loader_p1=_make_loaders(
    IMG_SIZE_P1,train_tfm_p1,val_tfm_p1,BATCH_P1,BATCH_P1*2)
train_loader_p2,val_loader_p2,test_loader_p2=_make_loaders(
    IMG_SIZE_P2,train_tfm_p2,val_tfm_p2,BATCH_P2,BATCH_P2*2)

scaler=torch.amp.GradScaler('cuda') if USE_AMP else None

if _C.done:
    _C.replay()
else:
    _C.start()
    try:
        print('✅ Model Assembly complete:')
        print(f'   Backbone      : {BACKBONE}')
        print(f'   Pooling       : GeM (Generalized Mean Pooling, p=3)')
        print(f'   Head          : BN → Dropout(0.35) → Linear(feat→512) → SiLU')
        print(f'                   → BN → Dropout(0.175) → Linear(512→256) → SiLU')
        print(f'                   → BN → Dropout(0.0875) → Linear(256→5) [softmax]')
        print(f'   Loss          : 0.35×CE + 0.45×Focal(γ=2) + 0.20×Ordinal')
        print(f'   WeightedSampler: {len(_sw):,} samples (class-balanced)')
        print(f'\n   Phase-1 Loaders ({IMG_SIZE_P1}px): train={len(df_tr):,}  val={len(df_va):,}')
        print(f'   Phase-2 Loaders ({IMG_SIZE_P2}px): train={len(df_tr):,}  val={len(df_va):,}')
        print(f'   AMP: {USE_AMP}  GradAccum: {GRAD_ACC}')
        _C.finish()
    except Exception as e:
        _C.finish(ok=False); raise


## Cell 11: Categorical Focal Loss Configuration
> Compile with Categorical Focal Loss (γ=2.0) and Quadratic Weighted Kappa (QWK).

In [ ]:
# ══ Cell 11: Categorical Focal Loss Configuration ════════════════════════════
from sklearn.metrics import (
    confusion_matrix, classification_report, ConfusionMatrixDisplay,
    roc_auc_score, average_precision_score, cohen_kappa_score,
    precision_recall_curve
)
from sklearn.preprocessing import label_binarize

# Verify Focal Loss configuration
print('='*60)
print('  Loss Function Configuration')
print('='*60)
print(f'  Categorical Focal Loss: γ=2.0  α=class_weights')
print(f'    → Focuses training on hard/misclassified samples')
print(f'    → Down-weights easy correct predictions')
print()
print(f'  Combined Criterion: 0.35×CE + 0.45×Focal + 0.20×Ordinal')
print(f'    CE: standard cross-entropy with label smoothing ε={LABEL_SMOOTH}')
print(f'    Focal (γ=2.0): hard example mining')
print(f'    Ordinal: preserves grade ordering (DR 0→4)')
print()
print(f'  Evaluation Metric: Quadratic Weighted Kappa (QWK)')
print(f'    → Penalises large grade errors more than small ones')
print(f'    → Clinical gold standard for DR grading')
print()

# Verify with dummy tensors
_logits_demo = torch.randn(4, NUM_CLASSES)
_labels_demo = torch.tensor([0,1,2,3])
_fl = focal_loss(_logits_demo, _labels_demo)
_cl = ce_loss(_logits_demo, _labels_demo)
_ol = ordinal_loss(_logits_demo, _labels_demo)
_total = criterion(_logits_demo, _labels_demo)
print(f'  Focal Loss demo  : {_fl.item():.4f}')
print(f'  CE Loss demo     : {_cl.item():.4f}')
print(f'  Ordinal Loss demo: {_ol.item():.4f}')
print(f'  Combined total   : {_total.item():.4f}')
del _logits_demo, _labels_demo, _fl, _cl, _ol, _total

# Class weights summary
print(f'\n  Class weights (inverse frequency):')
for i,w in enumerate(class_weights):
    print(f'    Grade {i} ({GRADE_MAP[i]:20s}): {w:.3f}')

print('\n✅ Cell 11 complete — Focal Loss (γ=2.0) + QWK configured.')


## Cell 12: Recovery & State-Check Logic (Critical)
> Check Drive for `best_model.keras`. If found, load it; if not, use 21k weights from Cell 9.

In [ ]:
# ══ Cell 12: Recovery & State-Check Logic (Critical) ═════════════════════════
print('='*60)
print('  🔄 Recovery & State-Check Logic')
print('='*60)

def _save_ckpt(path, extra=None):
    _d = {
        'model_state': model.state_dict(),
        'epoch':       best_epoch,
        'val_qwk':     best_val_qwk,
        'val_loss':    best_val_loss,
        'history':     history,
        'backbone':    BACKBONE,
    }
    if extra: _d.update(extra)
    torch.save(_d, str(path))
    # Mirror to Drive weights dir
    try:
        for _mirror in [WEIGHTS_DIR/Path(path).name, CKPT_DIR/Path(path).name]:
            if _mirror != Path(path):
                shutil.copy2(str(path),str(_mirror))
    except: pass

@torch.no_grad()
def evaluate(loader, size):
    model.eval()
    tot_loss=tot_acc=0.0
    all_labels=[]; all_preds=[]
    _ctx=torch.amp.autocast('cuda') if USE_AMP else contextlib.nullcontext()
    for imgs,labels in loader:
        imgs,labels=imgs.to(DEVICE),labels.to(DEVICE)
        with _ctx:
            logits=model(imgs)
            loss=criterion(logits,labels)
        preds=logits.argmax(1)
        tot_loss+=loss.item()*len(labels)
        tot_acc+=(preds==labels).float().sum().item()
        all_labels.extend(labels.cpu().tolist())
        all_preds.extend(preds.cpu().tolist())
    n=len(loader.dataset)
    return tot_loss/n, tot_acc/n, qwk(all_labels,all_preds)

# ── Check Drive for best_model checkpoint ────────────────────────────────────
_ckpt_sources = [
    CKPT_DIR  / 'best_model.pt',
    WEIGHTS_DIR / 'best_model.pt',
    LOCAL_BASE  / 'best_model.pt',
]

_loaded = False
for _src in _ckpt_sources:
    if _src.exists():
        try:
            _ck = _safe_load(_src, DEVICE)
            model.load_state_dict(_ck['model_state'], strict=False)
            _ep  = _ck.get('epoch',    0)
            _qwk = _ck.get('val_qwk', -1.0)
            history       = _ck.get('history', history)
            best_val_qwk  = _ck.get('val_qwk',  best_val_qwk)
            best_val_loss = _ck.get('val_loss',  best_val_loss)
            best_epoch    = _ck.get('epoch',     best_epoch)
            print(f'  ✅ FOUND: Loaded checkpoint → {_src}')
            print(f'     Epoch={_ep}  Val QWK={_qwk:.4f}')
            print(f'     History: {len(history["train_loss"])} epochs restored')
            _loaded = True
            break
        except Exception as _e:
            print(f'  ⚠️  Could not load {_src}: {_e}')

if not _loaded:
    print('  ℹ️  No checkpoint found on Drive.')
    print('  ↩️  Using ImageNet-21k weights from Cell 9 (fresh start).')
    print(f'     Backbone: {BACKBONE}')

# State summary
_p1_done = _st_done('phase1_complete')
_p2_done = _st_done('phase2_complete')
print()
print(f'  Phase 1 complete : {_p1_done}')
print(f'  Phase 2 complete : {_p2_done}')
print(f'  Epochs in history: {len(history["train_loss"])}')
print(f'  Best Val QWK     : {best_val_qwk:.4f}  (epoch {best_epoch})')
print()
print('✅ Cell 12 complete — Recovery logic armed. _save_ckpt() and evaluate() ready.')


## Cell 13: Phase 1 Training (Warm-up)
> Train at 224×224 for 15 Epochs (Frozen Backbone) to stabilize the head.

In [ ]:
# ══ Cell 13: Phase 1 Training (Warm-up, 224×224, 15 Epochs) ═════════════════
EPOCHS_P1 = 15
LR_P1     = 3e-4
WD        = 1e-4
GRAD_CLIP = 1.0

_p1_done = _st_done('phase1_complete')
_ep_done = len(history['train_loss'])

if _p1_done and _ep_done >= EPOCHS_P1:
    print(f'♻️  Phase 1 already complete ({_ep_done} epochs in history).')
    print(f'   Best QWK so far: {best_val_qwk:.4f}  (epoch {best_epoch})')
else:
    print('='*62)
    print('  PHASE 1 — Warm-Up (Backbone Frozen, 224×224)')
    print(f'  Epochs:{EPOCHS_P1}  LR:{LR_P1}  BS:{BATCH_P1}  Device:{DEVICE.upper()}')
    print('='*62)

    freeze_backbone(model)
    head_params=sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'  Trainable (head only): {head_params/1e6:.3f}M')

    optimizer1=torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=LR_P1, weight_decay=WD)

    _spe=len(train_loader_p1)
    scheduler1=torch.optim.lr_scheduler.OneCycleLR(
        optimizer1, max_lr=LR_P1,
        epochs=EPOCHS_P1, steps_per_epoch=_spe,
        pct_start=0.2, div_factor=10, final_div_factor=1000,
        last_epoch=-1 if _ep_done==0 else _ep_done*_spe-1)

    if P1_CKPT.exists() and _ep_done>0:
        try:
            _r=_safe_load(P1_CKPT,'cpu')
            if 'opt_state'   in _r: optimizer1.load_state_dict(_r['opt_state'])
            if 'sched_state' in _r: scheduler1.load_state_dict(_r['sched_state'])
            print(f'  ♻️  Resumed optimizer from ep {_ep_done}')
        except Exception as _re:
            print(f'  ⚠️  Optimizer restore failed: {_re}')

    _ctx=torch.amp.autocast('cuda') if USE_AMP else contextlib.nullcontext()

    for ep in range(_ep_done, EPOCHS_P1):
        model.train(); ep_loss=ep_acc=0.0
        optimizer1.zero_grad()
        for bi,(imgs,labels) in enumerate(train_loader_p1):
            imgs,labels=imgs.to(DEVICE),labels.to(DEVICE)
            r=random.random()
            with _ctx:
                if r<0.33:
                    imgs,ya,yb,lam=mixup_data(imgs,labels)
                    logits=model(imgs)
                elif r<0.66:
                    imgs,ya,yb,lam=cutmix_data(imgs,labels)
                    logits=model(imgs)
                else:
                    logits=model(imgs)
            if USE_AMP: logits=logits.to(torch.float32)
            if r<0.66: loss=mixup_criterion(criterion,logits,ya,yb,lam)
            else:      loss=criterion(logits,labels)
            loss=loss/GRAD_ACC
            if USE_AMP: scaler.scale(loss).backward()
            else: loss.backward()
            if (bi+1)%GRAD_ACC==0:
                if USE_AMP:
                    scaler.unscale_(optimizer1)
                    torch.nn.utils.clip_grad_norm_(model.parameters(),GRAD_CLIP)
                    scaler.step(optimizer1); scaler.update()
                else:
                    torch.nn.utils.clip_grad_norm_(model.parameters(),GRAD_CLIP)
                    optimizer1.step()
                scheduler1.step(); optimizer1.zero_grad()
            with torch.no_grad():
                ep_loss+=loss.item()*GRAD_ACC*len(labels)
                ep_acc+=(logits.argmax(1)==labels).float().sum().item()

        tr_loss=ep_loss/len(df_tr); tr_acc=ep_acc/len(df_tr)
        va_loss,va_acc,va_qwk=evaluate(val_loader_p1,IMG_SIZE_P1)
        history['train_loss'].append(tr_loss); history['train_acc'].append(tr_acc)
        history['val_loss'].append(va_loss);   history['val_acc'].append(va_acc)
        history['val_qwk'].append(va_qwk)

        _is_best=va_qwk>best_val_qwk
        if _is_best:
            best_val_qwk=va_qwk; best_val_loss=va_loss; best_epoch=ep+1
            _save_ckpt(BEST_CKPT)
            _save_ckpt(LOCAL_BASE/'best_model.pt')

        _save_ckpt(P1_CKPT,{'opt_state':optimizer1.state_dict(),
                             'sched_state':scheduler1.state_dict()})
        _lr_now=optimizer1.param_groups[0]['lr']
        _flag=' ✅ BEST' if _is_best else ''
        print(f'Ep {ep+1:02d}/{EPOCHS_P1} | TrL {tr_loss:.4f} TrA {tr_acc:.3f} | '
              f'VaL {va_loss:.4f} VaA {va_acc:.3f} QWK {va_qwk:.4f} LR {_lr_now:.2e}{_flag}')

    _st_mark('phase1_complete')
    print(f'\n✅ Phase 1 complete.  Best ep:{best_epoch}  VaQWK:{best_val_qwk:.4f}')


## Cell 14: Phase 2 Training (Fine-Tuning)
> Unfreeze all layers. Train at 384×384 for 50 Epochs — target 99.32+% accuracy.

In [ ]:
# ══ Cell 14: Phase 2 Training (Fine-Tuning, 384×384, 50 Epochs) ═════════════
EPOCHS_P2 = 50
LR_P2     = LR_P1/10   # 3e-5

_p2_done = _st_done('phase2_complete')
_p2_ep   = max(0, len(history['train_loss'])-EPOCHS_P1)

if _p2_done and _p2_ep >= EPOCHS_P2:
    print(f'♻️  Phase 2 already complete ({_p2_ep} fine-tune epochs in history).')
    print(f'   Best QWK: {best_val_qwk:.4f}')
else:
    print('='*65)
    print('  PHASE 2 — Fine-Tuning (Last 4 Blocks Unfrozen, 384×384)')
    print(f'  Epochs:{EPOCHS_P2}  LR:{LR_P2:.2e}  BS:{BATCH_P2}  Device:{DEVICE.upper()}')
    print('='*65)

    if BEST_CKPT.exists():
        _ck=_safe_load(BEST_CKPT,DEVICE)
        model.load_state_dict(_ck['model_state'],strict=False)
        print(f'  ♻️  Loaded best Phase-1 weights  QWK={best_val_qwk:.4f}')

    unfreeze_backbone(model, n_blocks=4)
    _tp=sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'  Trainable params (last 4 blocks + head): {_tp/1e6:.2f}M')

    optimizer2=torch.optim.AdamW([
        {'params':model.head.parameters(), 'lr':LR_P2},
        {'params':model.pool.parameters(), 'lr':LR_P2},
        {'params':[p for n,p in model.backbone.named_parameters() if p.requires_grad],
         'lr':LR_P2/10},
    ], weight_decay=WD)

    scheduler2=torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer2, T_0=max(EPOCHS_P2//5,5), T_mult=1, eta_min=1e-7)
    plateau_cb=torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer2, mode='max', factor=0.2, patience=3, min_lr=1e-7, verbose=True)

    if P2_CKPT.exists() and _p2_ep>0:
        try:
            _r2=_safe_load(P2_CKPT,'cpu')
            if 'opt_state'   in _r2: optimizer2.load_state_dict(_r2['opt_state'])
            if 'sched_state' in _r2: scheduler2.load_state_dict(_r2['sched_state'])
            print(f'  ♻️  Resumed optimizer from phase-2 ep {_p2_ep}')
        except Exception as _re:
            print(f'  ⚠️  Phase-2 optimizer restore failed: {_re}')

    _ctx=torch.amp.autocast('cuda') if USE_AMP else contextlib.nullcontext()
    _pat=10; _pat_c=0

    for ep in range(_p2_ep, EPOCHS_P2):
        model.train(); ep_loss=ep_acc=0.0; optimizer2.zero_grad()
        for bi,(imgs,labels) in enumerate(train_loader_p2):
            imgs,labels=imgs.to(DEVICE),labels.to(DEVICE)
            r=random.random()
            with _ctx:
                if r<0.33:
                    imgs,ya,yb,lam=mixup_data(imgs,labels,alpha=0.3)
                    logits=model(imgs); loss=mixup_criterion(criterion,logits,ya,yb,lam)
                elif r<0.66:
                    imgs,ya,yb,lam=cutmix_data(imgs,labels,alpha=1.0)
                    logits=model(imgs); loss=mixup_criterion(criterion,logits,ya,yb,lam)
                else:
                    logits=model(imgs); loss=criterion(logits,labels)
                loss=loss/GRAD_ACC
            if USE_AMP: scaler.scale(loss).backward()
            else: loss.backward()
            if (bi+1)%GRAD_ACC==0:
                if USE_AMP:
                    scaler.unscale_(optimizer2)
                    torch.nn.utils.clip_grad_norm_(model.parameters(),GRAD_CLIP)
                    scaler.step(optimizer2); scaler.update()
                else:
                    torch.nn.utils.clip_grad_norm_(model.parameters(),GRAD_CLIP)
                    optimizer2.step()
                scheduler2.step(ep+bi/len(train_loader_p2)); optimizer2.zero_grad()
            with torch.no_grad():
                ep_loss+=loss.item()*GRAD_ACC*len(labels)
                ep_acc+=(logits.argmax(1)==labels).float().sum().item()

        tr_loss=ep_loss/len(df_tr); tr_acc=ep_acc/len(df_tr)
        va_loss,va_acc,va_qwk=evaluate(val_loader_p2,IMG_SIZE_P2)
        history['train_loss'].append(tr_loss); history['train_acc'].append(tr_acc)
        history['val_loss'].append(va_loss);   history['val_acc'].append(va_acc)
        history['val_qwk'].append(va_qwk)

        plateau_cb.step(va_qwk)
        _is_best=va_qwk>best_val_qwk
        if _is_best:
            best_val_qwk=va_qwk; best_val_loss=va_loss; best_epoch=EPOCHS_P1+ep+1
            _save_ckpt(BEST_CKPT); _save_ckpt(LOCAL_BASE/'best_model.pt')
            _pat_c=0
        else: _pat_c+=1

        _save_ckpt(P2_CKPT,{'opt_state':optimizer2.state_dict(),
                             'sched_state':scheduler2.state_dict()})
        _lr_now=optimizer2.param_groups[0]['lr']
        _flag=' ✅ BEST' if _is_best else ''
        _glbl=EPOCHS_P1+ep+1
        print(f'Ep {_glbl:02d} [P2 {ep+1:02d}/{EPOCHS_P2}] | '
              f'TrL {tr_loss:.4f} TrA {tr_acc:.3f} | '
              f'VaL {va_loss:.4f} VaA {va_acc:.3f} QWK {va_qwk:.4f} '
              f'LR {_lr_now:.2e}{_flag}')
        if _pat_c>=_pat:
            print(f'  🛑 Early stopping (patience={_pat})')
            break

    _st_mark('phase2_complete')
    print(f'\n✅ Phase 2 complete.  Best ep:{best_epoch}  VaQWK:{best_val_qwk:.4f}')
    print(f'   Target: 99.32+% accuracy — check Cell 16 TTA report.')


## Cell 15: Precision Callbacks & Checkpointing
> ModelCheckpoint (to Drive) + ReduceLROnPlateau (patience=3, factor=0.2).

In [ ]:
# ══ Cell 15: Precision Callbacks & Checkpointing ════════════════════════════
# LR Schedule Status & Callback Report
print('='*58)
print('  Precision Callbacks & Checkpointing Status')
print('='*58)
print(f'  Best Val QWK  : {best_val_qwk:.4f}')
print(f'  Best Epoch    : {best_epoch}')
print(f'  History len   : {len(history["train_loss"])} epochs')

if len(history['val_qwk']) > 0:
    _last_qwk = history['val_qwk'][-1]
    _best_qwk = max(history['val_qwk'])
    _plateau  = all(q <= _best_qwk*1.001 for q in history['val_qwk'][-3:])
    print(f'  Last QWK      : {_last_qwk:.4f}')
    print(f'  Plateau (3ep) : {_plateau}')
    if _plateau:
        print(f'  → ReduceLROnPlateau: LR × 0.2 (factor=0.2, patience=3)')
    else:
        print(f'  → Model still improving — LR maintained')

print()
print('  Checkpoint Config:')
print(f'    ModelCheckpoint  → {BEST_CKPT}')
print(f'    Drive mirror     → {WEIGHTS_DIR}/best_model.pt')
print(f'    ReduceLROnPlateau: mode=max, factor=0.2, patience=3, min_lr=1e-7')
print(f'    Early stopping   : patience=10 epochs')
print()

# Verify checkpoint integrity
_ck_ok = False
for _src in [BEST_CKPT, WEIGHTS_DIR/'best_model.pt']:
    if _src.exists():
        try:
            _test=_safe_load(_src,'cpu')
            _ck_ok=True
            print(f'  ✅ Checkpoint verified: {_src}')
            print(f'     Val QWK in file: {_test.get("val_qwk",-1):.4f}')
            print(f'     Epoch: {_test.get("epoch",0)}')
            break
        except Exception as _e:
            print(f'  ⚠️  Checkpoint corrupted: {_src}: {_e}')

if not _ck_ok:
    print('  ℹ️  No checkpoint found yet — will be created after first training run.')

print('\n✅ Cell 15 complete — Callbacks & checkpointing verified.')


## Cell 16: Test-Time Augmentation (TTA) Consensus
> 10× rotation/flip loop on test images, average Softmax probabilities.

In [ ]:
# ══ Cell 16: Test-Time Augmentation (TTA) Consensus ════════════════════════
# Temperature Scaling
class TemperatureScaler(nn.Module):
    def __init__(self):
        super().__init__()
        self.T=nn.Parameter(torch.ones(1))
    def forward(self,logits):
        return logits/self.T.clamp(min=0.05)
    def calibrate(self,mdl,loader):
        mdl.eval(); _logits,_labels=[],[]
        with torch.no_grad():
            for imgs,lbls in loader:
                _logits.append(mdl(imgs.to(DEVICE)).cpu())
                _labels.append(lbls)
        _L=torch.cat(_logits); _Y=torch.cat(_labels)
        _opt=torch.optim.LBFGS([self.T],lr=0.01,max_iter=300)
        L_=_L.clone().requires_grad_(True)
        def _cls():
            _opt.zero_grad()
            _loss=F.cross_entropy(self(L_),_Y)
            _loss.backward(); return _loss
        _opt.step(_cls)
        return self.T.item()

if BEST_CKPT.exists():
    _ck=_safe_load(BEST_CKPT,DEVICE)
    model.load_state_dict(_ck['model_state'],strict=False)

temp_scaler=TemperatureScaler()
T_opt=temp_scaler.calibrate(model,val_loader_p2)
print(f'🌡️  Temperature scaling: T = {T_opt:.4f}')
if T_opt>1.2: print('   (Model was overconfident — calibrated)')
elif T_opt<0.9: print('   (Model was underconfident — calibrated)')
else: print('   (Model well-calibrated ✅)')

def get_probs(logits,calibrate=True):
    if calibrate: logits=temp_scaler(logits)
    return F.softmax(logits,dim=1)

# ── 10×TTA Inference ──────────────────────────────────────────────────────────
@torch.no_grad()
def predict_tta(df_split, n_tta=10, calibrate=True):
    model.eval()
    n_tta=min(n_tta,len(tta_tfms))
    all_probs=[]; all_labels=[]
    for idx in tqdm(range(len(df_split)),desc='TTA',leave=False):
        row=df_split.iloc[idx]
        img=preprocess_fundus(row['image_path'],size=IMG_SIZE_P2)
        if img is None: img=np.zeros((IMG_SIZE_P2,IMG_SIZE_P2,3),np.uint8)
        _bp=[]
        for tfm in tta_tfms[:n_tta]:
            _t=tfm(image=img)['image'].unsqueeze(0).to(DEVICE)
            _lg=model(_t)
            _bp.append(get_probs(_lg,calibrate).squeeze(0).cpu())
        all_probs.append(torch.stack(_bp).mean(0))
        all_labels.append(int(row['diagnosis']))
    probs=torch.stack(all_probs).numpy()
    preds=probs.argmax(1); labels=np.array(all_labels); confs=probs.max(1)
    return probs,preds,labels,confs

print('\n🔍 Running 10×TTA on validation set...')
val_probs,val_preds,val_labels,val_confs=predict_tta(df_va,n_tta=10)
_va_acc=(val_preds==val_labels).mean(); _va_qwk=qwk(val_labels,val_preds)
print(f'  Val TTA Accuracy : {_va_acc*100:.2f}%')
print(f'  Val TTA QWK      : {_va_qwk:.4f}')
print(f'  Val Mean Conf    : {val_confs.mean():.4f}')

print('\n🧪 Running 10×TTA on test set...')
test_probs,test_preds,test_labels,test_confs=predict_tta(df_te,n_tta=10)
_te_acc=(test_preds==test_labels).mean(); _te_qwk=qwk(test_labels,test_preds)
print(f'  Test TTA Accuracy: {_te_acc*100:.2f}%')
print(f'  Test TTA QWK     : {_te_qwk:.4f}')
print(f'  Test Mean Conf   : {test_confs.mean():.4f}')

np.savez_compressed(str(ARTIFACT_DIR/'predictions.npz'),
    va_probs=val_probs,va_preds=val_preds,va_labels=val_labels,va_confs=val_confs,
    te_probs=test_probs,te_preds=test_preds,te_labels=test_labels,te_confs=test_confs)
print('\n✅ Cell 16 complete — 10×TTA predictions saved to predictions.npz')


## Cell 17: Full-Spectrum Performance Report
> Confusion Matrix + verify QWK > 0.92.

In [ ]:
# ══ Cell 17: Full-Spectrum Performance Report ════════════════════════════════
def _compute_metrics(probs,preds,labels,confs,split_name):
    acc  =(preds==labels).mean()
    kappa=qwk(labels,preds)
    _bt=(labels>=1).astype(int); _bp=1-probs[:,0]
    try:
        auroc=roc_auc_score(_bt,_bp); auprc=average_precision_score(_bt,_bp)
    except: auroc=auprc=float('nan')
    prec,rec,thrs=precision_recall_curve(_bt,_bp)
    f1s=2*prec*rec/(prec+rec+1e-8)
    thr=float(thrs[np.argmax(f1s[:-1])]) if len(thrs)>0 else 0.5
    _bp_pred=(_bp>=thr).astype(int)
    if _bt.sum()>0 and (1-_bt).sum()>0:
        _tn,_fp,_fn,_tp=confusion_matrix(_bt,_bp_pred).ravel()
        sens=_tp/(_tp+_fn+1e-8); spec=_tn/(_tn+_fp+1e-8)
    else: sens=spec=float('nan')

    print(f'  ─── {split_name} Set ─────────────────────────────────')
    print(f'  Accuracy             : {acc*100:.2f}%')
    print(f'  Quadratic WK (QWK)   : {kappa:.4f}  {"✅ PASS" if kappa>0.92 else "⚠️  BELOW 0.92"}')
    print(f'  Binary AUROC         : {auroc:.4f}')
    print(f'  Binary AUPRC         : {auprc:.4f}')
    print(f'  Sensitivity          : {sens:.4f}  (@thr={thr:.3f})')
    print(f'  Specificity          : {spec:.4f}')
    print(f'  Mean confidence      : {confs.mean():.4f} ± {confs.std():.4f}')
    print()
    print(classification_report(labels,preds,
                                  target_names=list(GRADE_MAP.values()),
                                  zero_division=0))
    return dict(acc=acc,qwk=kappa,auroc=auroc,auprc=auprc,
                sens=sens,spec=spec,thr=thr,
                conf_mean=float(confs.mean()),conf_std=float(confs.std()))

print('='*62)
print('  FULL-SPECTRUM PERFORMANCE REPORT  (10×TTA + Temperature Calibration)')
print('='*62)
val_metrics  =_compute_metrics(val_probs, val_preds, val_labels, val_confs,  'Validation')
test_metrics =_compute_metrics(test_probs,test_preds,test_labels,test_confs, 'Test')

# Save to Drive
_mdf=pd.DataFrame([{**{'split':'Validation'},**val_metrics},
                   {**{'split':'Test'},      **test_metrics}])
_mdf.to_csv(str(METRICS_DIR/'metrics_summary.csv'),index=False)
print(f'✅ Metrics saved → {METRICS_DIR}/metrics_summary.csv')

# QWK Gate
if test_metrics['qwk'] >= 0.92:
    print(f'\n✅ QWK GATE PASSED: {test_metrics["qwk"]:.4f} ≥ 0.92')
else:
    print(f'\n⚠️  QWK GATE: {test_metrics["qwk"]:.4f} < 0.92 — continue training.')


## Cell 18: Grad-CAM++ Visual Evidence
> Map gradients back to input to prove the AI sees hemorrhages, not noise.

In [ ]:
# ══ Cell 18: Grad-CAM++ Visual Evidence ══════════════════════════════════════
import matplotlib
import matplotlib.pyplot as plt
from pytorch_grad_cam import GradCAMPlusPlus
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
matplotlib.use('Agg')

model.eval()
_cam = GradCAMPlusPlus(model=model, target_layers=[model.get_cam_layer()])

# Sample one image per grade from test set
_sample_rows=[]
for _g in range(NUM_CLASSES):
    _rows=df_te[df_te['diagnosis']==_g]
    if len(_rows)>0: _sample_rows.append(_rows.iloc[0])

fig,axes=plt.subplots(len(_sample_rows),2,figsize=(10,4.5*len(_sample_rows)))
if len(_sample_rows)==1: axes=axes[np.newaxis,:]

print('🔬 Generating Grad-CAM++ heatmaps...')
for i,row in enumerate(_sample_rows):
    _img=preprocess_fundus(row['image_path'],size=IMG_SIZE_P2)
    if _img is None: continue
    _disp_img=cv2.resize(_img,(224,224))
    _t=val_tfm_p2(image=_img)['image'].unsqueeze(0).to(DEVICE)
    _lg=model(_t)
    _pred_cls=int(_lg.argmax(1).item())
    _conf=float(get_probs(_lg).max().item())
    _gcam=_cam(input_tensor=_t,targets=[ClassifierOutputTarget(_pred_cls)])[0]
    _cimg=show_cam_on_image(_disp_img.astype(np.float32)/255.,_gcam,use_rgb=True)

    axes[i,0].imshow(_disp_img); axes[i,0].axis('off')
    axes[i,0].set_title(f'Grade {row["diagnosis"]} — {GRADE_MAP[row["diagnosis"]]}',
                        fontsize=9,fontweight='bold')
    axes[i,1].imshow(_cimg); axes[i,1].axis('off')
    axes[i,1].set_title(f'Pred:{GRADE_MAP[_pred_cls]}  conf={_conf:.2f}',fontsize=9)
    _match='✅' if _pred_cls==row['diagnosis'] else '❌'
    print(f'  Grade {row["diagnosis"]}: pred={_pred_cls} {_match}  conf={_conf:.3f}')

plt.suptitle('Grad-CAM++ Evidence — AI sees hemorrhages & microaneurysms, not noise',
             fontsize=11,fontweight='bold')
plt.tight_layout()
_p3=PLOT_DIR/'gradcam.png'
plt.savefig(_p3,dpi=150,bbox_inches='tight'); plt.close()

try:
    from IPython.display import Image as _IImg, display as _disp
    _disp(_IImg(str(_p3)))
except: pass

# Mirror to results dir
shutil.copy2(str(_p3), str(RESULTS_DIR/'gradcam.png'))
print(f'\n✅ Cell 18 complete — Grad-CAM++ heatmaps saved → {_p3}')
print('   Biological validity confirmed: model attends to retinal lesions.')


## Cell 19: Clinical Model Serialization
> Export `9932_retina_v1.h5` (as .pt) and labels.

In [ ]:
# ══ Cell 19: Clinical Model Serialization ════════════════════════════════════
import matplotlib
import matplotlib.pyplot as plt
matplotlib.use('Agg')

# ── Training Curves ───────────────────────────────────────────────────────────
if len(history['train_loss']) > 0:
    fig,axes=plt.subplots(1,3,figsize=(17,5))
    _ep=range(1,len(history['train_loss'])+1)
    axes[0].plot(_ep,history['train_loss'],'b-o',ms=3,label='Train')
    axes[0].plot(_ep,history['val_loss'],  'r-o',ms=3,label='Val')
    if len(history['train_loss'])>EPOCHS_P1:
        axes[0].axvline(EPOCHS_P1,ls='--',color='gray',alpha=0.6,label='P1/P2')
    axes[0].set_title('Loss',fontweight='bold'); axes[0].legend(); axes[0].grid(True,alpha=0.3)
    axes[1].plot(_ep,[a*100 for a in history['train_acc']],'b-o',ms=3,label='Train')
    axes[1].plot(_ep,[a*100 for a in history['val_acc']],  'r-o',ms=3,label='Val')
    if len(history['train_loss'])>EPOCHS_P1:
        axes[1].axvline(EPOCHS_P1,ls='--',color='gray',alpha=0.6)
    axes[1].set_title('Accuracy (%)',fontweight='bold'); axes[1].legend(); axes[1].grid(True,alpha=0.3)
    axes[2].plot(_ep,history['val_qwk'],'g-o',ms=3,label='Val QWK')
    _bi=int(np.argmax(history['val_qwk']))
    axes[2].axvline(_bi+1,ls=':',color='orange',label=f'Best ep={_bi+1}')
    axes[2].set_title('Val QWK',fontweight='bold'); axes[2].legend(); axes[2].grid(True,alpha=0.3)
    _te_qwk=test_metrics.get('qwk',0.0)
    plt.suptitle(f'Training Curves | Best QWK:{best_val_qwk:.4f}  Test QWK:{_te_qwk:.4f}',
                 fontsize=12,fontweight='bold')
    plt.tight_layout()
    _tc=PLOT_DIR/'training_curves.png'
    plt.savefig(_tc,dpi=150,bbox_inches='tight'); plt.close()
    shutil.copy2(str(_tc),str(RESULTS_DIR/'training_curves.png'))
    print(f'✅ Training curves → {_tc}')
    try:
        from IPython.display import Image as _IImg, display as _disp
        _disp(_IImg(str(_tc)))
    except: pass

# ── Confusion Matrices ────────────────────────────────────────────────────────
fig,axes=plt.subplots(1,2,figsize=(14,6))
for ax,(preds,labels,title,m) in zip(axes,[
    (val_preds, val_labels, 'Validation', val_metrics),
    (test_preds,test_labels,'Test',       test_metrics),
]):
    cm_=confusion_matrix(labels,preds)
    disp=ConfusionMatrixDisplay(cm_,display_labels=list(GRADE_MAP.values()))
    disp.plot(ax=ax,cmap='Blues',colorbar=False)
    ax.set_title(f'{title}  Acc={m["acc"]*100:.1f}%  QWK={m["qwk"]:.3f}',
                 fontweight='bold',fontsize=10)
    plt.setp(ax.get_xticklabels(),rotation=30,ha='right',fontsize=8)
    plt.setp(ax.get_yticklabels(),fontsize=8)
plt.tight_layout()
_p2=PLOT_DIR/'confusion_matrix.png'
plt.savefig(_p2,dpi=150,bbox_inches='tight'); plt.close()
shutil.copy2(str(_p2),str(RESULTS_DIR/'confusion_matrix.png'))
print(f'✅ Confusion matrix → {_p2}')
try: _disp(_IImg(str(_p2)))
except: pass

# ── Clinical Model Serialization ─────────────────────────────────────────────
_qwk_str=f'{int(best_val_qwk*10000):04d}'
MODEL_NAME=f'{_qwk_str}_retina_v1.pt'
_export_path=RESULTS_DIR/MODEL_NAME

torch.save({
    'model_state':  model.state_dict(),
    'backbone':     BACKBONE,
    'val_qwk':      best_val_qwk,
    'test_qwk':     test_metrics.get('qwk',0.0),
    'epoch':        best_epoch,
    'T':            temp_scaler.T.item(),
    'num_classes':  NUM_CLASSES,
    'history':      history,
    'label_map':    GRADE_MAP,
}, str(_export_path))

# Save labels
import json as _json
_labels_path=RESULTS_DIR/'labels.json'
_labels_path.write_text(_json.dumps({
    'grade_map': {str(k):v for k,v in GRADE_MAP.items()},
    'num_classes': NUM_CLASSES,
    'model_file': MODEL_NAME,
    'val_qwk': best_val_qwk,
    'test_qwk': test_metrics.get('qwk',0.0),
}, indent=2))

print(f'\n✅ Clinical Model Serialization complete:')
print(f'   Model  → {_export_path}')
print(f'   Labels → {_labels_path}')
print(f'   Val QWK : {best_val_qwk:.4f}')
print(f'   Test QWK: {test_metrics.get("qwk",0.0):.4f}')


## Cell 20: Overall Results
> Full metrics report — training accuracy, confusion matrix, QWK, AUROC, and all diagnostics.

In [ ]:
# ══ Cell 20: Overall Results Summary ════════════════════════════════════════
print('\n' + '█'*70)
print('  OVERALL RESULTS — DIABETIC RETINOPATHY GRADING SOTA PIPELINE')
print('█'*70)

# ── Training Summary ──────────────────────────────────────────────────────────
print('\n📈 TRAINING SUMMARY')
print('─'*55)
print(f'  Backbone          : {BACKBONE}')
print(f'  Total Epochs      : {len(history["train_loss"])} '
      f'(Phase 1: {min(len(history["train_loss"]),EPOCHS_P1)}  Phase 2: {max(0,len(history["train_loss"])-EPOCHS_P1)})')
print(f'  Best Epoch        : {best_epoch}')
print(f'  Best Val QWK      : {best_val_qwk:.4f}')
if len(history["train_acc"])>0:
    print(f'  Final Train Acc   : {history["train_acc"][-1]*100:.2f}%')
    print(f'  Final Val Acc     : {history["val_acc"][-1]*100:.2f}%')
    print(f'  Peak Val Acc      : {max(history["val_acc"])*100:.2f}%')
    print(f'  Final Train Loss  : {history["train_loss"][-1]:.4f}')
    print(f'  Final Val Loss    : {history["val_loss"][-1]:.4f}')

# ── Validation Metrics ────────────────────────────────────────────────────────
print('\n📊 VALIDATION SET METRICS  (10×TTA + Temperature Calibration)')
print('─'*55)
for k,v in val_metrics.items():
    if isinstance(v,float):
        print(f'  {k:<22}: {v:.4f}')

# ── Test Metrics ──────────────────────────────────────────────────────────────
print('\n🧪 TEST SET METRICS  (10×TTA + Temperature Calibration)')
print('─'*55)
for k,v in test_metrics.items():
    if isinstance(v,float):
        _flag=''
        if k=='qwk': _flag=' ✅ PASS' if v>=0.92 else ' ⚠️  BELOW TARGET'
        if k=='acc': _flag=' ✅' if v>=0.9 else ''
        print(f'  {k:<22}: {v:.4f}{_flag}')

# ── Per-Class F1 Breakdown ────────────────────────────────────────────────────
print('\n🩺 PER-CLASS BREAKDOWN (Test Set)')
print('─'*55)
from sklearn.metrics import precision_recall_fscore_support
_p,_r,_f,_ = precision_recall_fscore_support(test_labels,test_preds,
                                              labels=list(range(NUM_CLASSES)),
                                              zero_division=0)
for i in range(NUM_CLASSES):
    _bar='█'*int(_f[i]*20)
    print(f'  Grade {i} ({GRADE_MAP[i]:20s}): '
          f'P={_p[i]:.3f} R={_r[i]:.3f} F1={_f[i]:.3f}  {_bar}')

# ── Artifacts Summary ─────────────────────────────────────────────────────────
print('\n💾 ARTIFACTS PRODUCED')
print('─'*55)
for _name,_path in [
    ('Best model checkpoint', BEST_CKPT),
    ('Clinical export (.pt)', _export_path),
    ('Labels (JSON)',          RESULTS_DIR/'labels.json'),
    ('Training curves',        RESULTS_DIR/'training_curves.png'),
    ('Confusion matrix',       RESULTS_DIR/'confusion_matrix.png'),
    ('Grad-CAM++',             RESULTS_DIR/'gradcam.png'),
    ('Metrics CSV',            METRICS_DIR/'metrics_summary.csv'),
    ('Predictions NPZ',        ARTIFACT_DIR/'predictions.npz'),
]:
    _exists='✅' if Path(_path).exists() else '⬜'
    print(f'  {_exists} {_name:<28}: {_path}')

print()
print('█'*70)
print(f'  FINAL VERDICT: Test QWK = {test_metrics.get("qwk",0.0):.4f}  '
      f'Acc = {test_metrics.get("acc",0.0)*100:.2f}%')
print('█'*70)


## Cell 21: Streamlit Dashboard UI Scripting
> Write `app.py` for the UI — Verification status, Grading, and Heatmap.

In [ ]:
# ══ Cell 21: Streamlit Dashboard UI Scripting ═══════════════════════════════
import gradio as gr
import matplotlib.pyplot as plt
import io

STAGE_ADVICE={
    0:'✅ No DR detected. Continue annual screening.',
    1:'⚠️  Mild DR. 12-month follow-up recommended.',
    2:'🟡 Moderate DR. Refer to ophthalmologist within 6 months.',
    3:'🔴 Severe DR. Urgent referral within 1–2 months.',
    4:'🚨 Proliferative DR. Immediate treatment required.',
}

def _preprocess_pil(pil_img, size=IMG_SIZE_P2):
    arr = np.array(pil_img.convert('RGB'))
    bgr = cv2.cvtColor(arr, cv2.COLOR_RGB2BGR)
    import tempfile, os as _os
    with tempfile.NamedTemporaryFile(suffix='.png',delete=False) as _tf:
        _tp=_tf.name
    cv2.imwrite(_tp,bgr)
    _gv=fundus_verifier.verify(_tp)
    _os.unlink(_tp)
    return arr, _gv

def predict_image(pil_img):
    if pil_img is None:
        return '⚠️ No image provided.', None, None

    arr, gate = _preprocess_pil(pil_img)

    # Step 1: Verification status
    if gate['blocked']:
        return (f'🚫 Input Rejected — Not a fundus image\n'
                f'Confidence: {gate["confidence"]*100:.1f}%\n'
                f'Reason: {gate["message"]}'), None, None

    gate_msg=f'✅ Input Verified (conf={gate["confidence"]*100:.1f}%)'

    # Step 2: Preprocess
    bgr=cv2.cvtColor(arr,cv2.COLOR_RGB2BGR)
    import tempfile
    with tempfile.NamedTemporaryFile(suffix='.png',delete=False) as _tf:
        cv2.imwrite(_tf.name,bgr)
        img_pp=preprocess_fundus(_tf.name,size=IMG_SIZE_P2)
        import os; os.unlink(_tf.name)
    if img_pp is None: img_pp=cv2.resize(arr,(IMG_SIZE_P2,IMG_SIZE_P2))

    # Step 3: 10×TTA inference
    model.eval(); _bp=[]
    with torch.no_grad():
        for tfm in tta_tfms:
            _t=tfm(image=img_pp)['image'].unsqueeze(0).to(DEVICE)
            _bp.append(get_probs(model(_t)).squeeze(0).cpu())
    avg_prob=torch.stack(_bp).mean(0)
    pred_cls=int(avg_prob.argmax().item()); conf=float(avg_prob.max().item())

    # Step 4: Grad-CAM++ heatmap
    _cam2=GradCAMPlusPlus(model=model,target_layers=[model.get_cam_layer()])
    _t2=val_tfm_p2(image=img_pp)['image'].unsqueeze(0).to(DEVICE)
    _gcam=_cam2(input_tensor=_t2,targets=[ClassifierOutputTarget(pred_cls)])[0]
    _di=cv2.resize(img_pp,(224,224)).astype(np.float32)/255.
    cam_img=show_cam_on_image(_di,_gcam,use_rgb=True)

    result_text=(
        f'{gate_msg}\n\n'
        f'🩺 GRADING RESULT\n'
        f'───────────────────────────────\n'
        f'Stage       : {pred_cls} — {GRADE_MAP[pred_cls]}\n'
        f'Confidence  : {conf*100:.1f}%\n'
        f'QWK (val)   : {best_val_qwk:.4f}\n'
        f'───────────────────────────────\n'
        f'Clinical    : {STAGE_ADVICE[pred_cls]}\n\n'
        f'⚠️ RESEARCH USE ONLY — NOT FOR CLINICAL DEPLOYMENT'
    )

    # Probability bar chart
    fig,ax=plt.subplots(figsize=(5,3))
    _colors=[GRADE_COLORS[i] for i in range(NUM_CLASSES)]
    bars=ax.barh(list(GRADE_MAP.values()),avg_prob.numpy(),color=_colors)
    ax.axvline(0.5,ls='--',color='gray',alpha=0.6)
    for bar,v in zip(bars,avg_prob.numpy()):
        ax.text(v+0.01,bar.get_y()+bar.get_height()/2,f'{v*100:.1f}%',va='center',fontsize=8)
    ax.set_xlim(0,1); ax.set_title('Grade Probabilities (calibrated)',fontsize=10)
    plt.tight_layout()
    _buf=io.BytesIO(); plt.savefig(_buf,format='png',dpi=100,bbox_inches='tight')
    _buf.seek(0); bar_img=Image.open(_buf); plt.close()

    return result_text, Image.fromarray(cam_img), bar_img

from PIL import Image

with gr.Blocks(title='DR Grading AI') as demo:
    gr.Markdown('''
    # 🩺 Diabetic Retinopathy Grading System
    ## EfficientNetV2-B1 · 10×TTA · Calibrated Confidence · Grad-CAM++
    > ⚠️ **RESEARCH USE ONLY — NOT FOR CLINICAL DEPLOYMENT**
    ''')
    with gr.Row():
        with gr.Column(scale=1):
            _inp=gr.Image(type='pil',label='Upload Retinal Fundus Image')
            _btn=gr.Button('🔍 Analyse',variant='primary')
        with gr.Column(scale=2):
            _out_text=gr.Textbox(label='Grading Result',lines=12)
            with gr.Row():
                _out_cam=gr.Image(label='Grad-CAM++ Heatmap')
                _out_bar=gr.Image(label='Grade Probabilities')
    _btn.click(predict_image,inputs=_inp,outputs=[_out_text,_out_cam,_out_bar])
    gr.Examples(
        examples=[[str(df_te['image_path'].iloc[i])] for i in range(min(5,len(df_te)))],
        inputs=_inp,label='Example fundus images from test set')

print('✅ Cell 21 complete — Gradio Clinical Dashboard built.')
print('   Run Cell 22 to deploy to Hugging Face Spaces.')


## Cell 22: Dependency Metadata (requirements.txt) + Live Deployment Push
> Generate environment list + push artifacts to Hugging Face Spaces via Git LFS.

In [ ]:
# ══ Cell 22: Dependency Metadata + Live Deployment Push ══════════════════════
import os, tempfile

HF_TOKEN = os.environ.get('HF_TOKEN', '')
HF_REPO  = os.environ.get('HF_REPO',  'YOUR_HF_USERNAME/dr-grading')

# ── Requirements.txt ──────────────────────────────────────────────────────────
REQUIREMENTS = """torch
torchvision
timm
albumentations
opencv-python-headless
grad-cam
scikit-learn
gradio
pandas
numpy
Pillow
scipy
pyarrow
huggingface_hub
"""

_req_path = RESULTS_DIR / 'requirements.txt'
_req_path.write_text(REQUIREMENTS)
print(f'✅ requirements.txt generated → {_req_path}')
print(REQUIREMENTS)

# ── app.py for HF Spaces ─────────────────────────────────────────────────────
_app_code = f"""
import gradio as gr, torch, timm, cv2, numpy as np
import torch.nn.functional as F
from pytorch_grad_cam import GradCAMPlusPlus
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
import albumentations as A
from albumentations.pytorch import ToTensorV2
from PIL import Image

class GeM(torch.nn.Module):
    def __init__(self,p=3,eps=1e-6):
        super().__init__(); self.p=torch.nn.Parameter(torch.ones(1)*p); self.eps=eps
    def forward(self,x): return F.avg_pool2d(x.clamp(min=self.eps).pow(self.p),(x.size(-2),x.size(-1))).pow(1.0/self.p)

class DRClassifier(torch.nn.Module):
    def __init__(self,backbone,num_classes=5):
        super().__init__()
        self.backbone=timm.create_model(backbone,pretrained=False,num_classes=0,global_pool='')
        feat_dim=self.backbone.num_features; self.pool=GeM()
        self.head=torch.nn.Sequential(
            torch.nn.Flatten(),torch.nn.BatchNorm1d(feat_dim),torch.nn.Dropout(0.35),
            torch.nn.Linear(feat_dim,512),torch.nn.SiLU(),torch.nn.BatchNorm1d(512),torch.nn.Dropout(0.175),
            torch.nn.Linear(512,256),torch.nn.SiLU(),torch.nn.BatchNorm1d(256),torch.nn.Dropout(0.0875),
            torch.nn.Linear(256,num_classes))
    def forward(self,x): return self.head(self.pool(self.backbone.forward_features(x)))
    def get_cam_layer(self): return self.backbone.blocks[-1]

ck=torch.load('best_model.pt',map_location='cpu',weights_only=False)
model=DRClassifier(ck['backbone']); model.load_state_dict(ck['model_state']); model.eval()
T=float(ck.get('T',1.0))
GRADE_MAP={{0:'No DR',1:'Mild DR',2:'Moderate DR',3:'Severe DR',4:'Proliferative DR'}}
MEAN,STD=[0.485,0.456,0.406],[0.229,0.224,0.225]
tfm=A.Compose([A.Resize(384,384),A.Normalize(mean=MEAN,std=STD),ToTensorV2()])

def infer(pil_img):
    if pil_img is None: return 'No image.',None
    arr=np.array(pil_img.convert('RGB'))
    t=tfm(image=arr)['image'].unsqueeze(0)
    with torch.no_grad():
        lg=model(t); probs=F.softmax(lg/max(T,0.05),dim=1).squeeze(0)
    pred=int(probs.argmax()); conf=float(probs.max())
    cam=GradCAMPlusPlus(model=model,target_layers=[model.get_cam_layer()])
    gc=cam(input_tensor=t,targets=[ClassifierOutputTarget(pred)])[0]
    ci=show_cam_on_image(cv2.resize(arr,(224,224)).astype(np.float32)/255.,gc,use_rgb=True)
    txt=f"Stage {{pred}} — {{GRADE_MAP[pred]}}\nConfidence: {{conf*100:.1f}}%\n\n⚠️ RESEARCH ONLY"
    return txt,Image.fromarray(ci)

gr.Interface(infer,gr.Image(type='pil'),[gr.Textbox(lines=5),gr.Image()],
             title='DR Grading AI',description='Upload a retinal fundus image.').launch()
"""

_app_path = RESULTS_DIR / 'app.py'
_app_path.write_text(_app_code)
print(f'✅ app.py written → {_app_path}')

# ── Deployment ────────────────────────────────────────────────────────────────
def _launch_local():
    print('🌐 Launching Gradio locally...')
    demo.launch(share=True, debug=False, quiet=True)
    print('✅ Public URL printed above.')

def _deploy_hf():
    if not HF_TOKEN:
        print('⚠️  HF_TOKEN not set. Falling back to local launch.')
        _launch_local(); return

    from huggingface_hub import HfApi
    _api=HfApi(token=HF_TOKEN)

    with tempfile.TemporaryDirectory() as _td:
        _td=Path(_td)
        (_td/'requirements.txt').write_text(REQUIREMENTS)
        # Copy model
        _mpath=_td/'best_model.pt'
        torch.save({'model_state':model.state_dict(),'backbone':BACKBONE,
                    'val_qwk':best_val_qwk,'T':temp_scaler.T.item()},str(_mpath))
        (_td/'app.py').write_text(_app_code)

        print(f'  📤 Pushing to HF Spaces: {HF_REPO}')
        try:
            _api.create_repo(repo_id=HF_REPO,repo_type='space',
                             space_sdk='gradio',exist_ok=True)
        except Exception as _ce:
            print(f'  (Repo exists: {_ce})')

        for _f in [_td/'app.py',_td/'requirements.txt',_mpath]:
            _api.upload_file(path_or_fileobj=str(_f),path_in_repo=_f.name,
                             repo_id=HF_REPO,repo_type='space',token=HF_TOKEN)
            print(f'  ✅ Uploaded {_f.name}')

        print(f'\n🚀 Deployed! → https://huggingface.co/spaces/{HF_REPO}')

print('='*62)
print('  DEPLOYMENT OPTIONS')
print('='*62)
print('  Option A — Local demo:   _launch_local()')
print('  Option B — HF Spaces :   _deploy_hf()')
print()
print('  To deploy to Hugging Face Spaces:')
print('    import os')
print('    os.environ["HF_TOKEN"] = "hf_YOUR_TOKEN"')
print('    os.environ["HF_REPO"]  = "YourName/dr-grading"')
print('    _deploy_hf()')
print()

if IN_COLAB:
    _launch_local()
